## Runtime Diagnostics
Run this first after kernel restart to confirm the notebook is using the project environment and to get the active Spark UI URL.

In [1]:
import os
import sys
from pathlib import Path

# Resolve project root regardless of whether notebook cwd is project root or notebooks/.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent

# Force local Java/Hadoop runtime used by this project (avoids Java 24 incompatibility).
os.environ["JAVA_HOME"] = str(project_root / "jdk-17.0.2")
os.environ["HADOOP_HOME"] = str(project_root / "hadoop")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["PATH"] = (
    str(project_root / "jdk-17.0.2" / "bin")
    + os.pathsep
    + str(project_root / "hadoop" / "bin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

import pyspark
from pyspark.sql import SparkSession

print("Python executable:", sys.executable)
print("PySpark version:", pyspark.__version__)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

spark = (
    SparkSession.builder.appName("NotebookSession")
    .master("local[*]")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.use.daemon", "false")
    .config("spark.driver.extraJavaOptions", "-Duser.name=notebook")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Spark UI URL:", spark.sparkContext.uiWebUrl)

ImportError: cannot import name 'PySparkRuntimeError' from 'pyspark.errors' (unknown location)

# PySpark Big Data Learning
## GPS & Traffic Data Analysis from UK Government Sources

This notebook demonstrates PySpark fundamentals using real-world UK traffic data.

## 1. Initialize Spark Session

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, count, max as spark_max, min as spark_min

# Make src import work in local notebook execution.
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Reuse existing Spark session if already created by diagnostics cell.
spark = SparkSession.getActiveSession() or (
    SparkSession.builder
    .appName("BigDataLearning")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.python.use.daemon", "false")
    .config("spark.driver.extraJavaOptions", "-Duser.name=notebook")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")

## 2. Load GPS Data

In [ ]:
spark.sparkContext.uiWebUrl

In [ ]:
from pathlib import Path
from datetime import datetime
from src.data_fetcher import generate_sample_gps_data

# Safe conversion path on Windows: pandas -> CSV -> Spark read (JVM path)
def pandas_to_spark_via_csv(pdf, name_prefix="data"):
    cwd = Path.cwd()
    root = cwd if (cwd / "src").exists() else cwd.parent
    tmp_dir = root / "data" / "_tmp_notebook"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    csv_path = tmp_dir / f"{name_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
    pdf.to_csv(csv_path, index=False)
    return spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

# Generate and load sample GPS data
pdf = generate_sample_gps_data(10000)
gps_df = pandas_to_spark_via_csv(pdf, "gps")

print(f"Total records: {gps_df.count()}")
print("\nDataFrame Schema:")
gps_df.printSchema()

In [ ]:
# Replacement for the failing pandas -> Spark conversion cell
from pathlib import Path
from datetime import datetime
from src.data_fetcher import generate_sample_gps_data

# Safe conversion path on Windows: pandas -> CSV -> Spark read
# This avoids Python worker crashes seen with spark.createDataFrame(pandas_df).
def pandas_to_spark_via_csv(pdf, name_prefix="data"):
    cwd = Path.cwd()
    root = cwd if (cwd / "src").exists() else cwd.parent
    tmp_dir = root / "data" / "_tmp_notebook"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    csv_path = tmp_dir / f"{name_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
    pdf.to_csv(csv_path, index=False)
    return spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

pdf = generate_sample_gps_data(10000)
gps_df = pandas_to_spark_via_csv(pdf, "gps")

print(f"Total records: {gps_df.count()}")
print("\nDataFrame Schema:")
gps_df.printSchema()

## 3. Explore Data

In [ ]:
# Show sample records
print("Sample GPS Data:")
gps_df.show(10)



In [ ]:
# Show statistics
print("\nData Statistics:")
gps_df.describe(['speed_kmh', 'latitude', 'longitude']).show()

## 4. Basic Transformations

In [ ]:
# Filter: Get high-speed events (> 70 km/h)
high_speed = gps_df.filter(col('speed_kmh') > 70)
print(f"High-speed events (>70 km/h): {high_speed.count()}")
print("\nSample high-speed records:")
high_speed.show(10)

In [ ]:
# Select specific columns
subset = gps_df.select('timestamp', 'vehicle_id', 'location', 'speed_kmh')
print("Selected columns:")
subset.show(10)

## 5. Aggregations & Grouping

In [ ]:
# Average speed by location
print("Average Speed by Location:")
gps_df.groupBy('location') \
    .agg({
        'speed_kmh': 'avg',
        'vehicle_id': 'count'
    }) \
    .withColumnRenamed('avg(speed_kmh)', 'avg_speed') \
    .withColumnRenamed('count(vehicle_id)', 'vehicle_count') \
    .sort('avg_speed', ascending=False) \
    .show()

In [ ]:
# Speed statistics by location
print("Speed Statistics by Location:")
gps_df.groupBy('location').agg(
    avg('speed_kmh').alias('avg_speed'),
    spark_min('speed_kmh').alias('min_speed'),
    spark_max('speed_kmh').alias('max_speed'),
    count('speed_kmh').alias('count')
).show()

## 6. Complex Queries with SQL

In [ ]:
# Register as temporary table
gps_df.createOrReplaceTempView("gps_data")

# Run SQL queries
result = spark.sql("""
SELECT 
    location,
    COUNT(*) as total_records,
    ROUND(AVG(speed_kmh), 2) as avg_speed,
    MAX(speed_kmh) as max_speed
FROM gps_data
GROUP BY location
ORDER BY avg_speed DESC
""")

print("Speed Analysis by Location (SQL):")
result.show()

## 7. Performance Analysis

In [ ]:
# Find vehicles with high average speed
print("Top 10 Vehicles by Average Speed:")
spark.sql("""
SELECT 
    vehicle_id,
    location,
    ROUND(AVG(speed_kmh), 2) as avg_speed,
    COUNT(*) as num_records
FROM gps_data
GROUP BY vehicle_id, location
ORDER BY avg_speed DESC
LIMIT 10
""").show()

## 8. Load UK Government Data

In [ ]:
from src.data_fetcher import UKGovDataFetcher

# Ensure helper exists even if cells were run out of order
if "pandas_to_spark_via_csv" not in globals():
    from pathlib import Path
    from datetime import datetime
    def pandas_to_spark_via_csv(pdf, name_prefix="data"):
        cwd = Path.cwd()
        root = cwd if (cwd / "src").exists() else cwd.parent
        tmp_dir = root / "data" / "_tmp_notebook"
        tmp_dir.mkdir(parents=True, exist_ok=True)
        csv_path = tmp_dir / f"{name_prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.csv"
        pdf.to_csv(csv_path, index=False)
        return spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

fetcher = UKGovDataFetcher()

print("Fetching TfL Traffic Data...")
traffic_pdf = fetcher.fetch_tfl_traffic_data()

if not traffic_pdf.empty:
    traffic_df = pandas_to_spark_via_csv(traffic_pdf, "traffic")
    print(f"\nTraffic Incidents: {traffic_df.count()}")
    traffic_df.show(20)
else:
    print("No traffic data available at this time")

In [ ]:
print("\nFetching TfL Line Status...")
line_pdf = fetcher.fetch_tfl_line_status()

if not line_pdf.empty:
    line_df = pandas_to_spark_via_csv(line_pdf, "line_status")
    print(f"\nTube Line Status: {line_df.count()} lines")
    line_df.show(20)
else:
    print("No line status data available at this time")

## 9. Demonstrate Spark Execution Plan

In [ ]:
# Show execution plan
query = gps_df.groupBy('location').agg(avg('speed_kmh')).sort('location')

print("Logical Plan:")
print(query.explain())

print("\nResults:")
query.show()

## 10. Summary

In [ ]:
print("="*60)
print("SUMMARY - PySpark Big Data Learning")
print("="*60)

print(f"\n✓ Spark Session: Active")
print(f"✓ Total GPS Records Processed: {gps_df.count():,}")
print(f"✓ Locations Covered: {gps_df.select('location').distinct().count()}")
print(f"✓ Average Speed Overall: {gps_df.agg(avg('speed_kmh')).collect()[0][0]:.2f} km/h")
print(f"✓ Data Source: UK Gov (TfL) + Synthetic GPS")
print("\nNext Steps:")
print("1. Try modifying the queries above")
print("2. Increase data volume (change 10000 to larger number)")
print("3. Add windowing functions for time-series analysis")
print("4. Export results to Parquet format")
print("5. Monitor the Spark UI at http://localhost:4040")